In [2]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np, pandas as pd, geopandas as gpd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance

ROOT = Path('/content/drive/MyDrive/Logika Ui')
HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'
TAU = 0.05

tabel = pd.read_csv(HASIL / 'tabel_gabungan_nasional_semua_bulan.csv', parse_dates=['tanggal'])
gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')
pusat = gadm2.to_crs('EPSG:6933').geometry.centroid.to_crs('EPSG:4326')
koor = pd.DataFrame({'GID_2': gadm2['GID_2'].values, 'lat': pusat.y.values, 'lon': pusat.x.values})
tabel = tabel.merge(koor, on='GID_2', how='left')
hari = tabel['tanggal'].dt.dayofyear
tabel['doy_sin'] = np.sin(2 * np.pi * hari / 365.25)
tabel['doy_cos'] = np.cos(2 * np.pi * hari / 365.25)
tabel['kelas'] = np.where(tabel['hujan_rata'] * 6 >= 1, 2, np.where(tabel['frac_hujan'] >= TAU, 1, 0))

dasar = ['lat', 'lon', 'doy_sin', 'doy_cos']
era00 = [c for c in tabel.columns if c.endswith('_00')]
him = [c for c in tabel.columns if c.startswith('him_') and c not in ('him_n', 'him_f5')]
set_fitur = {'C': dasar + era00, 'D': dasar + era00 + him}

rng = np.random.default_rng(1)
latih = tabel[tabel['tahun'] != 2024]
latih = latih.iloc[rng.choice(len(latih), 300_000, replace=False)]
uji = tabel[(tabel['tahun'] == 2024) & tabel['him_n'].notna()]
uji = uji.iloc[rng.choice(len(uji), 60_000, replace=False)]

for nama, kolom in set_fitur.items():
    m = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=250,
                                       l2_regularization=1.0, early_stopping=False, random_state=1)
    m.fit(latih[kolom], latih['kelas'])
    pi = permutation_importance(m, uji[kolom], uji['kelas'], scoring='neg_log_loss',
                                n_repeats=3, random_state=1)
    print('MODEL', nama)
    print(pd.Series(pi.importances_mean, index=kolom).sort_values(ascending=False).head(10).round(4).to_string())

Mounted at /content/drive


/tmp/ipykernel_1881/227066197.py:13: DtypeWarning: Columns (58) have mixed types. Specify dtype option on import or set low_memory=False.
  tabel = pd.read_csv(HASIL / 'tabel_gabungan_nasional_semua_bulan.csv', parse_dates=['tanggal'])


MODEL C
tcwv_00     0.0508
lon         0.0350
sp_00       0.0239
lat         0.0227
q_500_00    0.0213
q_700_00    0.0177
tcc_00      0.0118
q_850_00    0.0111
doy_cos     0.0073
u_700_00    0.0071
MODEL D
him_f0      0.1071
lon         0.0446
lat         0.0217
tcwv_00     0.0201
sp_00       0.0132
q_700_00    0.0131
him_f3      0.0100
q_850_00    0.0099
u_700_00    0.0072
him_f4      0.0071


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = Path('/content/drive/MyDrive/Logika Ui')
if not ROOT.exists():
    from google.colab import drive
    drive.mount('/content/drive')
HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'
TAU = 0.05

semua = pd.read_csv(HASIL / 'tabel_gabungan_nasional_semua_bulan.csv', nrows=0).columns
kolom = ['tahun', 'hujan_rata', 'frac_hujan', 'him_n'] + (['run'] if 'run' in semua else [])
tabel = pd.read_csv(HASIL / 'tabel_gabungan_nasional_semua_bulan.csv', usecols=kolom)

kelas = np.where(tabel['hujan_rata'] * 6 >= 1, 2, np.where(tabel['frac_hujan'] >= TAU, 1, 0))
Y = np.eye(3)[kelas]
ada = tabel['him_n'].notna().to_numpy()

oof = np.load(HASIL / 'prob_oof_nasional.npz', allow_pickle=True)
klim = {str(k): oof[f'm{i}'] for i, k in enumerate(oof['nama'])}['klimatologi']
fin = np.load(HASIL / 'prob_final_nasional.npz')
P = {'C': fin['PC'], 'D': fin['PD']}
assert len(klim) == len(tabel) == len(P['C'])


def brier3(Pm, m):
    Pm = np.clip(Pm[m], 1e-6, 1)
    Pm = Pm / Pm.sum(axis=1, keepdims=True)
    return ((Pm - Y[m]) ** 2).sum(axis=1).mean()


def ukur(m):
    hasil = {'baris': int(m.sum())}
    for nama, Pm in P.items():
        hasil['skill_' + nama] = 1 - brier3(Pm, m) / brier3(klim, m)
        hasil['auc_K1_' + nama] = roc_auc_score(kelas[m] == 1, Pm[m, 1])
    hasil['selisih_skill_D_C'] = hasil['skill_D'] - hasil['skill_C']
    return hasil

Mounted at /content/drive


## A · Dengan dan tanpa Himawari

Pada baris tanpa Himawari, D sama dengan C karena cadangan, sehingga selisihnya nol. Baris **dengan** Himawari menunjukkan sumbangan Himawari yang sebenarnya.

In [ ]:
bagian = {'semua baris': np.ones(len(tabel), bool), 'dengan Himawari': ada, 'tanpa Himawari (D = C)': ~ada}
print(pd.DataFrame({k: ukur(m) for k, m in bagian.items()}).T.round(4).to_string())

per_tahun = {th: ukur(ada & (tabel['tahun'] == th).to_numpy()) for th in sorted(tabel['tahun'].unique())}
print('\nBARIS DENGAN HIMAWARI, PER TAHUN')
print(pd.DataFrame(per_tahun).T.round(4).to_string())

                            baris  skill_C  auc_K1_C  skill_D  auc_K1_D  selisih_skill_D_C
semua baris             1187441.0   0.1706    0.6863   0.2109    0.7058             0.0403
dengan Himawari         1068170.0   0.1708    0.6839   0.2156    0.7058             0.0448
tanpa Himawari (D = C)   119271.0   0.1683    0.7056   0.1683    0.7056             0.0000

BARIS DENGAN HIMAWARI, PER TAHUN
         baris  skill_C  auc_K1_C  skill_D  auc_K1_D  selisih_skill_D_C
2020  175314.0   0.1614    0.6878   0.2159    0.7146             0.0545
2021  174835.0   0.1637    0.6841   0.2188    0.7113             0.0551
2022  165734.0   0.1713    0.6667   0.2206    0.6911             0.0493
2023   65623.0   0.2923    0.7551   0.3383    0.7758             0.0461
2024  174835.0   0.1439    0.6935   0.1902    0.7153             0.0463
2025  174356.0   0.1604    0.6728   0.2041    0.6952             0.0437
2026  137473.0   0.1805    0.6719   0.1928    0.6835             0.0123


In [ ]:
if 'run' in tabel.columns:
    larian = {r: ukur((tabel['run'] == r).to_numpy()) for r in sorted(tabel['run'].dropna().unique())}
    print(pd.DataFrame(larian).T.round(4).to_string())
    print('\npersen kelas per larian')
    print(pd.crosstab(tabel['run'], kelas, normalize='index').round(3).to_string())
else:
    print('kolom run tidak ada di tabel; lewati bagian B')

       baris  skill_C  auc_K1_C  skill_D  auc_K1_D  selisih_skill_D_C
F  1005900.0   0.1703    0.6907   0.2142    0.7115             0.0440
L   181541.0   0.1724    0.6659   0.1925    0.6793             0.0202

persen kelas per larian
col_0      0      1      2
run                       
F      0.331  0.262  0.407
L      0.367  0.247  0.386
